# Beangrid data
Today's coffees (`coffee.csv`) and the full price history (`history/<year>.csv` + `catalog.csv`) since Nov 2023.

In [ ]:
import pandas as pd
from io import StringIO
from js import fetch

async def load(path):
    resp = await fetch(f'https://beangrid.mcconomy.org/{path}')
    return pd.read_csv(StringIO(await resp.text()))

coffee = await load('data/coffee.csv')
coffee.head()

In [ ]:
# Best value per kg among in-stock retail bags with trustworthy data
bags = coffee[coffee.available & ~coffee.suspect & coffee.grams.between(150, 600)]
bags.sort_values('ppk')[['roaster', 'title', 'size', 'price', 'ppk', 'origins', 'process', 'url']].head(20)

In [ ]:
# Geshas: price per kg vs bag size
geshas = coffee[coffee.available & coffee.title.str.contains('(?i)gei?sha')]
geshas.plot.scatter('grams', 'ppk')

In [ ]:
# Price history for any coffee: join the change log to the catalog
history = pd.concat([await load(f'data/history/{y}.csv') for y in (2025, 2026)])  # add 2023/2024 for more
catalog = await load('data/catalog.csv')
h = history.merge(catalog, on='key')
h['date'] = pd.to_datetime(h['date'])
example = h[h.title.str.contains('(?i)kenya') & h.listed.eq(1)]
example.pivot_table(index='date', columns='key', values='price').ffill().iloc[:, :8].plot(legend=False)